In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import anndata as ad
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
data = ad.read_h5ad(r'/home/jamin/condot/datasets/chemCPA/sciplex_complete_middle_subset_lincs_genes_v2.h5ad')
# data = ad.read_h5ad(r'/home/jamin/condot/datasets/chemCPA/sciplex_complete_middle_subset_v2.h5ad')

In [ ]:
# 定义 Dataset 类
class CellTypeDataset(Dataset):
    def __init__(self, data, split='train', device='cpu'):
        """
        初始化数据集，加载指定划分的数据，转换为 PyTorch 张量，并转移到指定设备
        :param data: AnnData 对象，包含稀疏矩阵和元信息
        :param split: 数据划分类型（'train', 'test', 'ood'）
        :param device: 设备类型（'cpu' 或 'cuda'）
        """
        self.data = data
        self.device = device
        
        # 根据 random_split 划分数据
        # self.idx = self.data.obs[self.data.obs['split_random'] == split].index
        self.idx = self.data.obs['split_random'] == split  # 布尔掩码
        # 将稀疏矩阵转换为稠密张量，并转移到指定设备
        self.X = torch.tensor(self.data.X[self.idx].toarray(), dtype=torch.float32, device=self.device)
        # 提取标签并转换为整数编码
        self.le = LabelEncoder()
        self.y = torch.tensor(
            self.le.fit_transform(self.data.obs.loc[self.idx, 'cell_type'].values),  # 使用 loc 筛选标签
            dtype=torch.long,
            device=self.device
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 定义模型
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        """
        简单的分类器，只有一层线性层
        """
        super(SimpleClassifier, self).__init__()
        self.fc = nn.Linear(input_dim, output_dim, bias=False)
    
    def forward(self, x):
        return self.fc(x)

# 定义训练函数
def train_model(model, train_loader, test_loader, criterion, optimizer, device, num_epochs=10):
    model.train()  # 设置为训练模式
    model.to(device)  # 将模型转移到指定设备
    for epoch in range(num_epochs):
        running_loss = 0.0
        for batch_idx, (x, y) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()

        current_lr = scheduler.get_last_lr()[0]
        # if current_lr >= 0.0001:
        # if epoch < 1:
        # 每个 epoch 更新学习率
        scheduler.step()

        # 打印当前学习率、损失函数值
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, LR: {current_lr:.6f}")

        print('------------test--------------')
        # 评估 OOD 数据
        evaluate_model(model, test_loader, device)
        
        print('------------ood--------------')
        # 评估 OOD 数据
        evaluate_model(model, ood_loader, device)


def evaluate_model(model, data_loader, device):
    model.eval()  # 设置为评估模式
    model.to(device)  # 确保模型在正确的设备上
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            _, predicted = torch.max(output.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    
    accuracy = 100 * correct / total
    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')
    conf_matrix = confusion_matrix(all_labels, all_preds)

    print(f'Accuracy: {accuracy:.2f}%')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1 Score: {f1:.4f}')
    print('Confusion Matrix:')
    print(conf_matrix)
    


In [ ]:
# 假设 data 是你的 AnnData 对象
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cuda:3')

# 加载数据集
train_dataset = CellTypeDataset(data, split='train', device=device)
test_dataset = CellTypeDataset(data, split='test', device=device)
ood_dataset = CellTypeDataset(data, split='ood', device=device)

# 创建 DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
ood_loader = DataLoader(ood_dataset, batch_size=32, shuffle=False)

In [ ]:
print(len(train_dataset))
print(len(test_dataset))
print(len(ood_dataset))

In [ ]:
# 初始化模型
input_dim = data.X.shape[1]  # 特征维度
output_dim = len(data.obs['cell_type'].unique())  # 类别数
model = SimpleClassifier(input_dim, output_dim)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)  # 加入L2正则化

# 添加学习率调度器
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.5)

In [ ]:
# 训练模型
train_model(model, train_loader, test_loader, criterion, optimizer, device, num_epochs=2)


In [ ]:
# 保存整个模型
torch.save(model, '/home/jamin/condot/notebooks/classifier/classifier_sciplex_lincs_genes.pth')
# 保存模型的权重矩阵
torch.save(model.state_dict(), '/home/jamin/condot/notebooks/classifier/classifier_weight_matrix_sciplex_lincs_genes.pth')